# Inference of Single Population Mean ($\mu$) when $\sigma^2 $ is unknown, for small sample size ($n < 30$)

## Standard T test Procedure

The test statistic is given by $$T_{CALC}=\dfrac{\bar{X}-\mu_0 }{\left(\dfrac{S }{\sqrt{n}}\right)}$$ which follows a $t$ distribution with $(n-1)$ degrees of freedom.

The code for a standard $t$ test for a single sample is provided below:

In [34]:
import numpy as np
import scipy.stats as stats
import pandas as pd

def T_test_1sample(sample_mean, sample_std, sample_size, mu_0, alternative, alpha):
    degrees_of_freedom = sample_size - 1
    def T_calc(sample_mean, sample_std, sample_size, mu_0):
        return (sample_mean - mu_0) / (sample_std / np.sqrt(sample_size))
    
    t_value = T_calc(sample_mean, sample_std, sample_size, mu_0)

#The table value    
    def T_table(alpha, alternative, degrees_of_freedom):
        if alternative == 'two sided':
            p_value = 2 * (1 - stats.t.cdf(abs(t_value),degrees_of_freedom))
            return stats.t.ppf(1 - alpha / 2, degrees_of_freedom), p_value
        elif alternative == 'greater':
            p_value = 1 - stats.t.cdf(t_value, degrees_of_freedom)
            return stats.t.ppf(1 - alpha, degrees_of_freedom), p_value
        elif alternative == 'less':
            p_value = stats.t.cdf(t_value, degrees_of_freedom)
            return stats.t.ppf(alpha, degrees_of_freedom), p_value
        else:
            raise ValueError('Type either "two sided", "greater" or "less" for the alternative')
            
    critical_value, p_value = T_table(alpha, alternative,degrees_of_freedom)
    
    result = {"T_calc": t_value, "T_table": critical_value, "P-value": p_value}
    result_table = pd.DataFrame([result], index=['Values'])
    
    styled_table = table = result_table.style.set_caption('T Test Results')
    
    return styled_table

In [35]:
# Test example: A demonstration of a use case

T_test_1sample(sample_mean=33.712, sample_std=0.798, sample_size=14, mu_0=33, alternative='greater', alpha=0.05)

,T_calc,T_table,P-value
Values,3.338421,1.770933,0.002669


### Example 1

Data: The data contains 29 data points of recorded density of the earth measured independently, in grams per cubic centimeter $(g/cm^3)$.

Known fact: It is known that the average density of the earth is 5.51 $g/cm^3$. (Reference: https://share.google/Lqll74uasWgKipqSr)

Objective: Let $\mu_0$ be the true average density of the earth. That is, $$\mu_0 = 5.51$$
    We want to test whether our dataset accurately reflects the true average density of the earth by testing whether our sample mean (sample average density) is statistically equal to the true mean (true average density).

In particular, we want to test:
$$H_{0}: \quad \mu = 5.51 \quad \text{vs} \quad \mu \neq 5.51 $$
at 95% confidence level.

In [36]:
#Loading the dataset

df = pd.read_excel("Measurements of Density of Earth.xlsx")

In [37]:
df.head()

,Density
0,5.50
1,5.61
2,4.88
3,5.07
4,5.26


In [38]:
df.describe()

,Density
count,29.000000
mean,5.454138
std,0.230039
min,4.880000
25%,5.300000
50%,5.460000
75%,5.610000
max,5.860000


In [39]:
print("Sample standard deviation is : ", df['Density'].std())
print("Sample mean is : ", df['Density'].mean())
print("Sample size is : ", df['Density'].count())

Sample standard deviation is :  0.23003908427822986
Sample mean is :  5.4541379310344835
Sample size is :  29


In [40]:
T_test_1sample(sample_mean = df['Density'].mean(), sample_std = df['Density'].std(), sample_size = df['Density'].count(), mu_0 = 5.51, alternative='two sided', alpha=0.05)

,T_calc,T_table,P-value
Values,-1.307719,2.048407,0.201605


**Interpretation**: We fail to reject the null hypothesis at $\alpha = 0.05$ and conclude that the sample does not provide sufficient evidence that the mean density differs from 5.51.

### Example 2

Data: We simulate 25 randomly distributed values with true mean $\mu_0 = 10.31$. For the purpose of simulation, we shall use a population standard deviation $\sigma = 1.20$. We will then assume that this population standard deviation is unknown, as it is required for standard t tests.

Objective: We want evidence that the true mean is greater than 10, statistically. Although by inspection it is clearly greater than 10. 

We want to test
$$H_{0}: \quad \mu \leq 10 \quad \text{vs} \quad \mu > 10 \quad \text{at} \quad \alpha=0.05 $$

In [56]:
#seed for reproducibility
np.random.seed(123)

sim_data = np.random.normal(loc = 10.31, scale = 1.20, size = 25)

print("Sample standard deviation is : ", sim_data.std())
print("Sample mean is : ", sim_data.mean())
print("Sample size is : ", len(sim_data))

Sample standard deviation is :  1.4481814849030292
Sample mean is :  10.47812334808561
Sample size is :  25


In [69]:
T_test_1sample(sample_mean = sim_data.mean(), sample_std = sim_data.std(), sample_size = len(sim_data), mu_0 = 10, alternative='greater', alpha=0.05)

,T_calc,T_table,P-value
Values,1.650772,1.710882,0.055906


**Interpretation:** The evidence from the sample is not strong enough to conclude that the true mean is greater than the hypothesized value (for the one-sided test). The result is close to significance $(p \approx 0.056)$, but still above the threshold, so we retain the null hypothesis at 95% confidence level.

## Confidence Interval on Mean $\mu$ for T test

* Two Sided CI
$$ \bar{x} - t_{n-1,\alpha \mathbin{/} 2}\dfrac{S }{\sqrt{n}} \leq \mu \leq \bar{x} + t_{n-1,\alpha \mathbin{/} 2}\dfrac{S }{\sqrt{n}}$$

* Upper One Sided
$$\mu \leq \bar{x} + t_{n-1,\alpha \mathbin{/} 2}\dfrac{S }{\sqrt{n}}$$

* Lower One Sided
$$ \mu \geq \bar{x} - t_{n-1,\alpha \mathbin{/} 2}\dfrac{S }{\sqrt{n}}$$

Now the code for confidence interval on single population $\mu $ with unknown population standard deviation $S$ under a Students $T$ test is given as follows

In [65]:
import numpy as np
import scipy.stats as stats
import pandas as pd

def T_confidence_interval_1mean(alpha,sample_mean,sample_std,sample_size,alternative):
    degrees_of_freedom = sample_size - 1
    if alternative == 'two sided':
        lower_bound = sample_mean - stats.t.ppf(1 - alpha/2, degrees_of_freedom)*((sample_std)/np.sqrt(sample_size))
        upper_bound = sample_mean + stats.t.ppf(1 - alpha/2, degrees_of_freedom)*((sample_std)/np.sqrt(sample_size))
        conf_interval = (lower_bound, upper_bound)
    
    elif alternative == 'greater':
        upper_bound = sample_mean + stats.t.ppf(1 - alpha, degrees_of_freedom)*((sample_std)/np.sqrt(sample_size))
        conf_interval = print("μ ≤ ", upper_bound)
    
    elif alternative == 'less':
        lower_bound = sample_mean - stats.t.ppf(1 - alpha, degrees_of_freedom)*((sample_std)/np.sqrt(sample_size))
        conf_interval = print("μ ≥  ",lower_bound)
    
    else:
        raise ValueError('Type either "two sided", "greater" or "less" for the alternative')
    
    return conf_interval

In [66]:
print("A 95% Confidence Interval on true population mean μ : ")
T_confidence_interval_1mean(alpha=0.05,sample_mean=5.448,sample_std=0.221,sample_size=29,alternative='two sided')

A 95% Confidence Interval on true population mean μ : 


(np.float64(5.36393609582069), np.float64(5.532063904179311))

In [67]:
T_confidence_interval_1mean(alpha=0.05,sample_mean=df['Density'].mean(),sample_std=df['Density'].std(),sample_size=df['Density'].count(),alternative='two sided')

(np.float64(5.366635743078549), np.float64(5.541640118990418))

## Power Calculation for T Tests

* $H_{0}:\mu = \mu_{0}$ vs $H_{0}:\mu \neq \mu_{0}$ 
$$\text{Power } = \phi \left[-t_{n-1, \alpha \mathbin{/}2} + \dfrac{(\mu_{0}- \mu_{a})\sqrt{n}}{s }\right] + \phi \left[-t_{n-1,\alpha \mathbin{/}2} + \dfrac{(\mu_{a}- \mu_{0})\sqrt{n}}{s }\right] $$

* $H_{0}:\mu \leq \mu_{0}$ vs $H_{0}:\mu > \mu_{0}$ 
$$\text{Power } = \phi \left[-t_{n-1, \alpha } - \dfrac{(\mu_{a}- \mu_{0})\sqrt{n}}{\sigma }\right] $$

* $H_{0}:\mu \geq \mu_{0}$ vs $H_{0}:\mu > \mu_{0}$ 
$$\text{Power } = 1 - \phi \left[-t_{n-1, \alpha } - \dfrac{(\mu_{a}- \mu_{0})\sqrt{n}}{\sigma }\right] $$

In [23]:
import numpy as np
import scipy.stats as stats
import pandas as pd

def T_power(alpha, mu_0, mu_A, sample_size, sample_std, alternative):
    degrees_of_freedom = sample_size - 1
    if alternative == 'two sided':
        t_power = stats.t.cdf(-stats.t.ppf(1 - alpha / 2, degrees_of_freedom) + ((mu_0 - mu_A)*np.sqrt(sample_size))/sample_std, degrees_of_freedom) + stats.t.cdf(-stats.t.ppf(1 - alpha / 2, degrees_of_freedom) + ((mu_A - mu_0)*np.sqrt(sample_size))/sample_std,degrees_of_freedom)
        return t_power
    elif alternative == 'greater':
        t_power = 1 - stats.t.cdf(stats.t.ppf(1 - alpha, degrees_of_freedom) - ((mu_A - mu_0)*np.sqrt(sample_size))/sample_std, degrees_of_freedom)
        return  t_power
    elif alternative == 'less':
        t_power = stats.t.cdf(-stats.t.ppf(1 - alpha, degrees_of_freedom) - ((mu_A - mu_0)*np.sqrt(sample_size))/sample_std, degrees_of_freedom)
        return  t_power
    else:
        raise ValueError('Type either "two sided", "greater" or "less" for the alternative')

In [25]:
T_power(alpha=0.1, mu_0=3, mu_A=3.3, sample_size=16, sample_std=0.5, alternative='greater')

0.8469089267685082